# 02 · Speech, offline

1. Local **Piper** voices speak the replies
2. **faster-whisper** hears them back and detects the language (round trip in five languages)
3. The **whole voice pipeline runs with every non-local network connection blocked**, as proof of the offline core

In [1]:
import json, os, pathlib, time, warnings
import pandas as pd
warnings.filterwarnings("ignore")
if pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")
os.environ.pop("ELEVENLABS_API_KEY", None)  # local Piper voice only: these notebooks demonstrate the offline core
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

from IPython.display import Audio, display
from coop_assistant.speech import voice

## 1. Offline voices (Piper)

In [2]:
clip_en, engine_en = voice.speak("Just so you know, this is demo data. In Nyeri, parchment coffee grade A "
                                  "is going for 120 shillings a kilo.", "en")
clip_sw, engine_sw = voice.speak("Bei ya marejeleo ya kahawa ya parchment daraja A huko Nyeri ni KES 120 kwa kilo.", "sw")
print("engines:", engine_en, engine_sw)
display(Audio(clip_en), Audio(clip_sw))

engines: piper piper


## 2. Round trip in five languages

Each question is spoken by Piper and transcribed by Whisper. This is a hard test: synthetic voices are robotic and the domain words are rare. Real speakers still need testing (see `docs/DATA_CARD.md`).

In [3]:
questions = {
    "en": "The buyer offered 105 shillings per kilo for grade A parchment coffee in Nyeri. Is that fair?",
    "hi": "व्यापारी ने न्येरी में ग्रेड ए पार्चमेंट कॉफ़ी के लिए 105 शिलिंग प्रति किलो बोला है। क्या यह सही है?",
    "sw": "Mnunuzi ametoa shilingi 105 kwa kilo kwa kahawa ya parchment daraja A huko Nyeri. Je, ni sawa?",
    "zh": "买家给Nyeri的A级羊皮纸咖啡豆每公斤出价105先令，这个价格合理吗？",
    "ko": "구매자가 Nyeri의 A등급 파치먼트 커피에 kg당 105실링을 제시했어요. 적당한가요?",
}
rows = []
for lang, text in questions.items():
    wav, _ = voice.speak(text, lang)
    start = time.time()
    heard, detected, prob = voice.transcribe(wav)
    rows.append({"spoken_lang": lang, "detected": detected, "confidence": round(prob, 2),
                 "lang_ok": detected == lang, "stt_seconds": round(time.time() - start, 1), "heard": heard})
pd.DataFrame(rows)

,spoken_lang,detected,confidence,lang_ok,stt_seconds,heard
0,en,en,1.00,True,9.7,The buyer offered 105 shillings per kilo for great apartment coffee in Nyeri. Is that fair?
1,hi,hi,0.98,True,7.4,व्या पारी ने नेरी में ग्रेद ए पार्च्मेंट कोफी के लिए एक सो पाच शिलिंपती किलो बोला है. क्या यह सही है?
2,sw,sw,0.86,True,6.0,"Nunuzi, ame tuwa shillindi miyemo jana tanu, kwa kilu kwa kahawa yu parchmenti na radya, ahu Konyeri, ye, nisawa."
3,zh,zh,1.00,True,6.3,"买家给乃的A鸡羊皮,纸咖啡,东美公镜出价150线令这个价格合理吗?"
4,ko,ko,1.00,True,8.4,"구매자가, Nyeri 위에 A등급 파치먼트, 커피에게 쥐당 105 샬링을 제시했어요. 적당한가요?"


## 3. Voice in → answer → voice out, with the internet blocked

In [4]:
import socket

class NoInternet:
    """Allow loopback (Ollama on localhost) only; any other connection attempt raises."""
    def __enter__(self):
        self.original, self.blocked = socket.socket.connect, []
        def guarded(sock, address):
            host = address[0] if isinstance(address, tuple) else address
            if host not in ("127.0.0.1", "::1", "localhost"):
                self.blocked.append(host)
                raise OSError(f"blocked: {host}")
            return self.original(sock, address)
        socket.socket.connect = guarded
        return self
    def __exit__(self, *exc):
        socket.socket.connect = self.original

from coop_assistant.core.store import connect
from coop_assistant.pipeline import Assistant

turns = ["The buyer offered 105 shillings per kilo for grade A parchment coffee in Nyeri.",
         "It is parchment coffee, grade A."]
with NoInternet() as net:
    bot = Assistant(connect(":memory:"))
    start = time.time()
    for said in turns:
        wav, _ = voice.speak(said, "en")
        heard, lang, _ = voice.transcribe(wav)
        r = bot.handle(heard, lang)
        reply_wav, reply_engine = voice.speak(r["reply"], r["lang"])
        print(f"caller said : {said}\nwhisper heard: {heard}\nstate       : "
              f"{r['decision']['state']}/{r['decision']['reason']}  evidence={r['decision']['evidence_id']}\n"
              f"reply ({reply_engine}): {r['reply']}\n")
        if r["decision"]["state"] != "CLARIFY":
            break
print(f"non-local connection attempts: {net.blocked or 'none'} | total {time.time() - start:.1f}s")
display(Audio(reply_wav))

caller said : The buyer offered 105 shillings per kilo for grade A parchment coffee in Nyeri.
whisper heard: The buyer offered 105 shillings per kilo for grade apartment coffee in Nyeri
state       : CLARIFY/missing  evidence=None
reply (piper): Please tell me the coffee type (cherry or parchment).



caller said : It is parchment coffee, grade A.
whisper heard: It is parchment coffee, grade A
state       : ANSWER/matched  evidence=DEMO-NYR-PARCH-A
reply (piper): Just so you know, this is demo data. According to Demo Co-op Nyeri's records from October 2, parchment coffee grade A in Nyeri is going for 120 shillings a kilo. So the 105 you were offered is 15 shillings below that. It's your call whether to sell, and do confirm the grade and moisture with your co-op.

non-local connection attempts: none | total 22.4s
